In [ ]:

!pip -q install umap-learn seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path


DATASET_DIR = Path('')
RESULTS_DIR = Path('')

RANDOM_STATE = 42
N_CLUSTERS = 24
PCA_COMPONENTS = 100
FIG_DPI = 600
PLOT_SAMPLE = 4000
SILHOUETTE_SAMPLE = 3000
NUM_WORKERS = 2


MAX_IMAGES_PER_SPLIT = None

RUN_GMM = True
RUN_UMAP = True
RUN_TSNE = True

FEATURE_CONFIGS = [
    {
        'name': 'DINOv2Small',
        'display_name': 'DINOv2-Small (ViT-S/14)',
        'kind': 'dinov2',
        'hub_model': 'dinov2_vits14',
        'image_size': 518,
        'batch_size': 32,
    },
    {
        'name': 'DINOv2B14',
        'display_name': 'DINOv2-B/14 (ViT-B/14)',
        'kind': 'dinov2',
        'hub_model': 'dinov2_vitb14',
        'image_size': 518,
        'batch_size': 16,
    },
]

for sub in ['figures', 'tables', 'cache']:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)

print('Dataset:', DATASET_DIR)
print('Results:', RESULTS_DIR)
print('Fixed number of clusters/classes:', N_CLUSTERS)

In [ ]:
import json
import math
import random
import textwrap
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision.transforms as T
from PIL import Image
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    accuracy_score,
    calinski_harabasz_score,
    classification_report,
    completeness_score,
    confusion_matrix,
    davies_bouldin_score,
    f1_score,
    fowlkes_mallows_score,
    homogeneity_score,
    normalized_mutual_info_score,
    precision_score,
    recall_score,
    silhouette_score,
    v_measure_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='paper')


FONT_SIZE_BOOST = 4
_font_keys = [
    'font.size',
    'axes.labelsize',
    'axes.titlesize',
    'xtick.labelsize',
    'ytick.labelsize',
    'legend.fontsize',
    'legend.title_fontsize',
    'figure.titlesize',
]
_base_font = float(plt.rcParams['font.size'])
for _key in _font_keys:
    _current = plt.rcParams.get(_key, _base_font)
    if isinstance(_current, str) or _current is None:
        _current = _base_font
    plt.rcParams[_key] = float(_current) + FONT_SIZE_BOOST

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
def save_fig(name, fig=None, dpi=FIG_DPI):
    fig = fig or plt.gcf()
    out = RESULTS_DIR / 'figures'
    out.mkdir(parents=True, exist_ok=True)
    for ext in ['png', 'svg', 'pdf']:
        fig.savefig(out / f'{name}.{ext}', dpi=dpi, bbox_inches='tight')


def save_table(df, name):
    out = RESULTS_DIR / 'tables'
    out.mkdir(parents=True, exist_ok=True)
    path = out / f'{name}.csv'
    df.to_csv(path, index=False)
    return path


def read_rgb(path):
    with Image.open(path) as im:
        return im.convert('RGB')


def image_exts():
    return {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}


def maybe_sample(data, n=PLOT_SAMPLE):
    if n is None or len(data) <= n:
        return data.copy()
    return data.sample(n=n, random_state=RANDOM_STATE).copy()


def short_label(label, width=15):
    return '\n'.join(textwrap.wrap(str(label), width=width, break_long_words=False))


def model_display_name(feature_name):
    for cfg in FEATURE_CONFIGS:
        if cfg['name'] == feature_name:
            return cfg['display_name']
    return feature_name

## 1. Load Dataset And Audit

In [ ]:
labels_path = DATASET_DIR / 'logs' / 'labels.csv'
if not labels_path.exists():
    raise FileNotFoundError(f'Missing labels.csv: {labels_path}')

df = pd.read_csv(labels_path)
required_cols = {'split', 'image_name', 'label', 'flat_image_relative_path'}
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f'labels.csv missing required columns: {missing_cols}')

df['image_path'] = df['flat_image_relative_path'].apply(lambda p: DATASET_DIR / p)
df['exists'] = df['image_path'].apply(lambda p: p.exists())
df['suffix_ok'] = df['image_path'].apply(lambda p: p.suffix.lower() in image_exts())

if not df['exists'].all():
    display(df.loc[~df['exists'], ['split', 'image_name', 'image_path']].head(30))
    raise FileNotFoundError('Some images listed in labels.csv do not exist.')

if not df['suffix_ok'].all():
    display(df.loc[~df['suffix_ok'], ['split', 'image_name', 'image_path']].head(30))
    raise ValueError('Some listed files do not have known image extensions.')

split_order = ['train', 'valid', 'test']
if MAX_IMAGES_PER_SPLIT is not None:
    df = (
        df.groupby('split', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), MAX_IMAGES_PER_SPLIT), random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )

df['split'] = pd.Categorical(df['split'], categories=split_order, ordered=True)
df = df.sort_values(['split', 'image_name']).reset_index(drop=True)

audit_rows = []
for split in split_order:
    split_dir = DATASET_DIR / split
    disk_images = [
        p.name for p in split_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_exts()
    ]
    log_images = df.loc[df['split'].astype(str) == split, 'image_name'].tolist()
    audit_rows.append({
        'split': split,
        'images_in_log': len(log_images),
        'images_on_disk': len(disk_images),
        'missing_from_disk': len(set(log_images) - set(disk_images)),
        'extra_on_disk': len(set(disk_images) - set(log_images)),
        'duplicate_names_in_log': int(pd.Series(log_images).duplicated().sum()),
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
save_table(audit_df, 'dataset_audit')
print('Total images:', len(df))
print('Unique labels:', df['label'].nunique())

In [ ]:
size_cache = RESULTS_DIR / 'cache' / 'image_sizes.csv'
if size_cache.exists() and MAX_IMAGES_PER_SPLIT is None:
    sizes = pd.read_csv(size_cache)
else:
    rows = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc='Reading image sizes'):
        with Image.open(row.image_path) as im:
            width, height = im.size
        rows.append({
            'split': row.split,
            'image_name': row.image_name,
            'width': width,
            'height': height,
        })
    sizes = pd.DataFrame(rows)
    if MAX_IMAGES_PER_SPLIT is None:
        sizes.to_csv(size_cache, index=False)

df = df.merge(sizes, on=['split', 'image_name'], how='left')
df['area'] = df['width'] * df['height']
df['aspect_ratio'] = df['width'] / df['height']

class_dist = df.groupby(['split', 'label'], observed=False).size().reset_index(name='count')
display(class_dist.head())
save_table(class_dist, 'class_distribution')

fig, ax = plt.subplots(figsize=(17, 7))
sns.barplot(data=class_dist, x='label', y='count', hue='split', ax=ax)
ax.set_title('Class Distribution by Split')
ax.set_xlabel('Label')
ax.set_ylabel('Cropped Images')
ax.tick_params(axis='x', rotation=75)
save_fig('dataset_class_distribution_by_split', fig)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=df, x='width', hue='split', bins=40, ax=axes[0], multiple='layer')
axes[0].set_title('Crop Width')
sns.histplot(data=df, x='height', hue='split', bins=40, ax=axes[1], multiple='layer')
axes[1].set_title('Crop Height')
sns.scatterplot(data=maybe_sample(df), x='width', y='height', hue='split', s=12, alpha=0.5, ax=axes[2])
axes[2].set_title('Crop Width vs Height')
save_fig('dataset_crop_size_distribution', fig)
plt.show()

In [ ]:
sample = maybe_sample(df, 36)
cols = 6
rows = math.ceil(len(sample) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 3 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis('off')
for ax, row in zip(axes, sample.itertuples(index=False)):
    ax.imshow(read_rgb(row.image_path))
    ax.set_title(short_label(row.label, 18), fontsize=12)
    ax.axis('off')
fig.suptitle('Random Cropped Dataset Samples', fontsize=20)
save_fig('dataset_random_crop_samples', fig)
plt.show()

## 2. Feature Extraction

In [ ]:
class ImagePathDataset(torch.utils.data.Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return self.transform(read_rgb(self.paths[idx]))


def cache_path(name):
    suffix = '' if MAX_IMAGES_PER_SPLIT is None else f'_sample{MAX_IMAGES_PER_SPLIT}'
    return RESULTS_DIR / 'cache' / f'{name}{suffix}.npz'


def load_cached_features(name):
    path = cache_path(name)
    if path.exists():
        data = np.load(path)
        print(f'Loaded cached {name}:', data['features'].shape)
        return data['features']
    return None


def save_cached_features(name, features):
    np.savez_compressed(cache_path(name), features=features.astype(np.float32))


def build_transform(image_size):
    return T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def build_feature_model(cfg):
    if cfg['kind'] == 'dinov2':
        return torch.hub.load('facebookresearch/dinov2', cfg['hub_model'])

    raise ValueError(cfg['kind'])


@torch.no_grad()
def extract_torch_features(cfg, paths):
    name = cfg['name']
    cached = load_cached_features(name)
    if cached is not None:
        return cached

    transform = build_transform(cfg['image_size'])
    dataset = ImagePathDataset(paths, transform)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=cfg['batch_size'],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = build_feature_model(cfg).to(DEVICE).eval()
    features = []

    for batch in tqdm(loader, desc=f'Extracting {cfg["display_name"]}'):
        out = model(batch.to(DEVICE, non_blocking=True))
        if isinstance(out, dict):
            out = out.get('x_norm_clstoken', next(iter(out.values())))
        features.append(out.detach().cpu().float().numpy())

    features = np.concatenate(features, axis=0).astype(np.float32)
    save_cached_features(name, features)

    del model
    torch.cuda.empty_cache()
    return features


paths = df['image_path'].tolist()
feature_sets = {}
for cfg in FEATURE_CONFIGS:
    feature_sets[cfg['name']] = extract_torch_features(cfg, paths)

feature_summary = pd.DataFrame([
    {
        'feature_set': cfg['name'],
        'feature_model': cfg['display_name'],
        'n_images': feature_sets[cfg['name']].shape[0],
        'embedding_dim': feature_sets[cfg['name']].shape[1],
        'image_size': cfg['image_size'],
        'batch_size': cfg['batch_size'],
    }
    for cfg in FEATURE_CONFIGS
])
display(feature_summary)
save_table(feature_summary, 'feature_summary')

## 3. PCA Preprocessing

In [ ]:
label_encoder = LabelEncoder()
df['label_id'] = label_encoder.fit_transform(df['label'])
class_names = label_encoder.classes_
n_classes = len(class_names)
print('Evaluation label count:', n_classes)
if n_classes != N_CLUSTERS:
    raise ValueError(f'Expected exactly {N_CLUSTERS} classes, but found {n_classes}: {list(class_names)}')

split_masks = {split: (df['split'].astype(str).values == split) for split in split_order}
y_all = df['label_id'].values

prepared = {}
pca_rows = []

for feature_name, X in feature_sets.items():
    train_mask = split_masks['train']
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X[train_mask])
    X_scaled = scaler.transform(X)

    n_components = min(PCA_COMPONENTS, X_train_scaled.shape[0] - 1, X_train_scaled.shape[1])
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    pca.fit(X_train_scaled)
    X_pca = pca.transform(X_scaled)

    prepared[feature_name] = {
        'scaler': scaler,
        'pca': pca,
        'X_pca': X_pca,
    }

    cumulative = np.cumsum(pca.explained_variance_ratio_)
    for i, (ratio, cum) in enumerate(zip(pca.explained_variance_ratio_, cumulative), start=1):
        pca_rows.append({
            'feature_set': feature_name,
            'component': i,
            'explained_variance_ratio': ratio,
            'cumulative_explained_variance': cum,
        })

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(np.arange(1, len(cumulative) + 1), cumulative, marker='o', linewidth=1)
    ax.axhline(0.95, color='red', linestyle='--', label='95% variance')
    ax.set_title(f'PCA Cumulative Explained Variance: {model_display_name(feature_name)}')
    ax.set_xlabel('PCA Components')
    ax.set_ylabel('Cumulative Explained Variance')
    ax.legend()
    save_fig(f'{feature_name}_pca_explained_variance', fig)
    plt.show()

pca_df = pd.DataFrame(pca_rows)
display(pca_df.head())
save_table(pca_df, 'pca_explained_variance_all_models')

## 4. 24-Cluster Setup

In [ ]:
selected_k_by_feature = {feature_name: int(N_CLUSTERS) for feature_name in prepared}

fixed_k_summary = pd.DataFrame([
    {
        'feature_set': feature_name,
        'feature_model': model_display_name(feature_name),
        'fixed_k': int(N_CLUSTERS),
        'selection_rule': 'fixed to 24 classes/clusters by experimental design',
    }
    for feature_name in prepared
])

display(fixed_k_summary)
save_table(fixed_k_summary, 'fixed_24_clusters_by_feature')

## 5. Clustering and Evaluation Helpers

In [ ]:
def cluster_label_mapping(y_true, y_cluster):
    labels = np.unique(y_true)
    clusters = np.unique(y_cluster)
    contingency = np.zeros((len(labels), len(clusters)), dtype=np.int64)
    label_to_row = {label: i for i, label in enumerate(labels)}
    cluster_to_col = {cluster: i for i, cluster in enumerate(clusters)}

    for label, cluster in zip(y_true, y_cluster):
        contingency[label_to_row[label], cluster_to_col[cluster]] += 1

    rows, cols = linear_sum_assignment(-contingency)
    mapping = {clusters[c]: labels[r] for r, c in zip(rows, cols) if contingency[r, c] > 0}

    for cluster in clusters:
        if cluster in mapping:
            continue
        mask = y_cluster == cluster
        if mask.any():
            values, counts = np.unique(y_true[mask], return_counts=True)
            mapping[cluster] = values[np.argmax(counts)]
        else:
            mapping[cluster] = labels[0]
    return mapping


def apply_cluster_mapping(y_cluster, mapping):
    return np.array([mapping.get(cluster, -1) for cluster in y_cluster], dtype=int)


def sampled_internal_metrics(X, clusters, sample_size=SILHOUETTE_SAMPLE):
    clusters = np.asarray(clusters)
    unique_clusters = np.unique(clusters)
    if len(unique_clusters) < 2 or len(unique_clusters) >= len(clusters):
        return {
            'silhouette': np.nan,
            'davies_bouldin': np.nan,
            'calinski_harabasz': np.nan,
        }

    if sample_size is not None and len(X) > sample_size:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(len(X), size=sample_size, replace=False)
        X_eval = X[idx]
        c_eval = clusters[idx]
        if len(np.unique(c_eval)) < 2:
            return {
                'silhouette': np.nan,
                'davies_bouldin': np.nan,
                'calinski_harabasz': np.nan,
            }
    else:
        X_eval = X
        c_eval = clusters

    return {
        'silhouette': float(silhouette_score(X_eval, c_eval)),
        'davies_bouldin': float(davies_bouldin_score(X_eval, c_eval)),
        'calinski_harabasz': float(calinski_harabasz_score(X_eval, c_eval)),
    }


def evaluate_clustering(feature_name, cluster_model_name, split, X, y_true, y_cluster, mapping, selected_k):
    y_pred = apply_cluster_mapping(y_cluster, mapping)
    internal = sampled_internal_metrics(X, y_cluster)
    return {
        'feature_set': feature_name,
        'feature_model': model_display_name(feature_name),
        'cluster_model': cluster_model_name,
        'split': split,
        'selected_k': int(selected_k),
        'n_samples': int(len(y_true)),
        'n_clusters_found': int(len(np.unique(y_cluster))),
        'ari': float(adjusted_rand_score(y_true, y_cluster)),
        'nmi': float(normalized_mutual_info_score(y_true, y_cluster)),
        'ami': float(adjusted_mutual_info_score(y_true, y_cluster)),
        'homogeneity': float(homogeneity_score(y_true, y_cluster)),
        'completeness': float(completeness_score(y_true, y_cluster)),
        'v_measure': float(v_measure_score(y_true, y_cluster)),
        'fowlkes_mallows': float(fowlkes_mallows_score(y_true, y_cluster)),
        'silhouette': internal['silhouette'],
        'davies_bouldin': internal['davies_bouldin'],
        'calinski_harabasz': internal['calinski_harabasz'],
        'hungarian_accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
    }


def make_clusterer(cluster_model_name, n_clusters):
    if cluster_model_name == 'KMeans':
        return KMeans(n_clusters=n_clusters, n_init=20, random_state=RANDOM_STATE)
    if cluster_model_name == 'MiniBatchKMeans':
        return MiniBatchKMeans(
            n_clusters=n_clusters,
            n_init=20,
            batch_size=1024,
            random_state=RANDOM_STATE,
        )
    if cluster_model_name == 'GMM':
        return GaussianMixture(
            n_components=n_clusters,
            covariance_type='diag',
            random_state=RANDOM_STATE,
            max_iter=300,
        )
    raise ValueError(cluster_model_name)

## 6. Main Unsupervised Runs

For each DINOv2 feature extractor, the notebook fits KMeans, MiniBatchKMeans, and GMM using the fixed K = 24 setup. The final per-feature visual section automatically uses the best test macro-F1 clusterer for that feature extractor.

In [ ]:
cluster_models = ['KMeans', 'MiniBatchKMeans']
if RUN_GMM:
    cluster_models.append('GMM')

all_metric_rows = []
cluster_outputs = {}

for feature_name, pack in prepared.items():
    X = pack['X_pca']
    selected_k = int(selected_k_by_feature[feature_name])
    X_train = X[split_masks['train']]
    y_train = y_all[split_masks['train']]

    print(f'\nFeature extractor: {model_display_name(feature_name)} | fixed K={selected_k}')

    for cluster_model_name in cluster_models:
        print(f'  Fitting {cluster_model_name}')
        clusterer = make_clusterer(cluster_model_name, selected_k)
        start = time.time()

        if cluster_model_name == 'GMM':
            clusterer.fit(X_train)
            clusters_all = clusterer.predict(X)
        else:
            clusterer.fit(X_train)
            clusters_all = clusterer.predict(X)

        fit_seconds = time.time() - start
        train_clusters = clusters_all[split_masks['train']]
        mapping = cluster_label_mapping(y_train, train_clusters)
        y_pred_all = apply_cluster_mapping(clusters_all, mapping)

        key = (feature_name, cluster_model_name)
        cluster_outputs[key] = {
            'clusterer': clusterer,
            'clusters': clusters_all,
            'predicted_labels': y_pred_all,
            'mapping': mapping,
            'selected_k': selected_k,
            'fit_seconds': fit_seconds,
        }

        for split in split_order:
            mask = split_masks[split]
            row = evaluate_clustering(
                feature_name=feature_name,
                cluster_model_name=cluster_model_name,
                split=split,
                X=X[mask],
                y_true=y_all[mask],
                y_cluster=clusters_all[mask],
                mapping=mapping,
                selected_k=selected_k,
            )
            row['fit_seconds'] = fit_seconds
            all_metric_rows.append(row)

metrics_df = pd.DataFrame(all_metric_rows)
display(metrics_df.sort_values(['split', 'macro_f1'], ascending=[True, False]))
save_table(metrics_df, 'clustering_metrics_all_splits')

test_ranked = (
    metrics_df[metrics_df['split'] == 'test']
    .sort_values(['macro_f1', 'ari', 'nmi'], ascending=False)
    .reset_index(drop=True)
)
display(test_ranked)
save_table(test_ranked, 'clustering_metrics_test_ranked')

In [ ]:
test_metrics_for_plot = test_ranked.copy()
paper_metrics = ['ari', 'nmi', 'v_measure', 'hungarian_accuracy', 'macro_f1', 'weighted_f1']

fig, ax = plt.subplots(figsize=(13, 7))
plot_df = test_metrics_for_plot.melt(
    id_vars=['feature_model', 'cluster_model'],
    value_vars=paper_metrics,
    var_name='metric',
    value_name='score',
)
plot_df['method'] = plot_df['feature_model'] + ' + ' + plot_df['cluster_model']
sns.barplot(data=plot_df, x='metric', y='score', hue='method', ax=ax)
ax.set_title('Test Metrics Across Feature Extractors and Clustering Models')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Method')
save_fig('overall_test_metrics_bar_all_methods', fig)
plt.show()

heat = test_metrics_for_plot.set_index(['feature_model', 'cluster_model'])[paper_metrics]
fig, ax = plt.subplots(figsize=(11, max(4.5, 0.45 * len(heat))))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='viridis', linewidths=0.4, ax=ax)
ax.set_title('Test Metric Heatmap for All Unsupervised Methods')
save_fig('overall_test_metrics_heatmap_all_methods', fig)
plt.show()

## 7. Per-Feature Extractor


In [ ]:
def class_id_mapping_table():
    table = pd.DataFrame({
        'label_id': np.arange(n_classes, dtype=int),
        'label': class_names,
        'label_wrapped': [short_label(x, 18).replace('\n', ' / ') for x in class_names],
    })
    save_table(table, 'class_id_mapping')
    return table


class_map_df = class_id_mapping_table()
display(class_map_df)


best_summary_rows = []
best_keys = {}
for feature_name in prepared.keys():
    sub = (
        metrics_df[(metrics_df['split'] == 'test') & (metrics_df['feature_set'] == feature_name)]
        .sort_values(['macro_f1', 'ari', 'nmi'], ascending=False)
        .reset_index(drop=True)
    )
    best = sub.iloc[0].to_dict()
    best_keys[feature_name] = (feature_name, best['cluster_model'])
    best_summary_rows.append(best)

best_summary_df = pd.DataFrame(best_summary_rows)
display(best_summary_df[[
    'feature_model',
    'cluster_model',
    'selected_k',
    'ari',
    'nmi',
    'v_measure',
    'hungarian_accuracy',
    'macro_f1',
    'weighted_f1',
]])
save_table(best_summary_df, 'best_clusterer_per_feature_on_test')

In [ ]:
def _count_annotations(matrix):
    annot = np.empty(matrix.shape, dtype=object)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = int(matrix[i, j])
            annot[i, j] = '' if value == 0 else str(value)
    return annot


def plot_confusion_matrix_hardware_style(cm, labels, prefix, split_name):
    annot = _count_annotations(cm)
    vmax = max(1, int(cm.max()))
    n_labels = len(labels)
    fig_w = max(18, 0.72 * n_labels + 8)
    fig_h = max(13, 0.56 * n_labels + 5)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(
        cm,
        annot=annot,
        fmt='',
        cmap='Blues',
        vmin=0,
        vmax=vmax,
        linewidths=0.45,
        linecolor='#f1f1f1',
        xticklabels=labels,
        yticklabels=labels,
        cbar=True,
        cbar_kws={'shrink': 0.92},
        ax=ax,
    )

    ax.set_title(f'Confusion Matrix ({split_name})', fontsize=22, pad=18)
    ax.set_xlabel('Predicted', fontsize=18, labelpad=18)
    ax.set_ylabel('True', fontsize=18, labelpad=18)
    ax.tick_params(axis='x', rotation=90, labelsize=13)
    ax.tick_params(axis='y', rotation=0, labelsize=13)

    threshold = vmax * 0.45
    for text in ax.texts:
        label = text.get_text()
        if not label:
            continue
        value = int(label)
        text.set_fontsize(12)
        text.set_color('white' if value >= threshold else '#222222')

    colorbar = ax.collections[0].colorbar
    if colorbar is not None:
        colorbar.ax.tick_params(labelsize=14)

    fig.tight_layout()
    save_fig(f'{prefix}_confusion_matrix_{split_name}_hardware_clean', fig)
    plt.show()


def confusion_outputs(y_true, y_pred, prefix, title, split_name='test'):
    label_ids = list(range(n_classes))
    cm = confusion_matrix(y_true, y_pred, labels=label_ids)
    row_sum = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sum, out=np.zeros_like(cm, dtype=float), where=row_sum != 0)
    cm_pct = cm_norm * 100.0

    count_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    pct_df = pd.DataFrame(cm_pct, index=class_names, columns=class_names)
    save_table(count_df.reset_index(names='true_label'), f'{prefix}_{split_name}_confusion_matrix_counts')
    save_table(pct_df.reset_index(names='true_label'), f'{prefix}_{split_name}_confusion_matrix_row_normalized_percent')

    plot_confusion_matrix_hardware_style(cm, list(class_names), prefix, split_name)

    id_table = class_map_df.copy()
    save_table(id_table, f'{prefix}_{split_name}_class_id_mapping_for_confusion_matrix')
    return cm, cm_pct


def top_confusions(cm, prefix, title, split_name='test', top_n=20):
    columns = [
        'true_label_id',
        'true_label',
        'predicted_label_id',
        'predicted_label',
        'count',
        'percent_of_true_class',
        'pair',
    ]
    rows = []
    for true_id, true_name in enumerate(class_names):
        total = cm[true_id].sum()
        if total == 0:
            continue
        for pred_id, pred_name in enumerate(class_names):
            if true_id == pred_id:
                continue
            count = int(cm[true_id, pred_id])
            if count == 0:
                continue
            rows.append({
                'true_label_id': true_id,
                'true_label': true_name,
                'predicted_label_id': pred_id,
                'predicted_label': pred_name,
                'count': count,
                'percent_of_true_class': 100.0 * count / total,
                'pair': f'{true_id}: {true_name} -> {pred_id}: {pred_name}',
            })

    conf_df = pd.DataFrame(rows, columns=columns)
    if not conf_df.empty:
        conf_df = conf_df.sort_values(['count', 'percent_of_true_class'], ascending=False)
    save_table(conf_df, f'{prefix}_{split_name}_top_confusions_table')
    if conf_df.empty:
        print(f'No off-diagonal confusions for {title} on {split_name}')
        return conf_df

    plot_df = conf_df.head(top_n).iloc[::-1].copy()
    plot_df['pair_short'] = plot_df['pair'].map(lambda x: short_label(x, 55))
    fig, ax = plt.subplots(figsize=(12, max(6, 0.42 * len(plot_df))))
    ax.barh(plot_df['pair_short'], plot_df['count'], color='#33658a')
    ax.set_title(f'Top {min(top_n, len(conf_df))} Confusions ({split_name}): {title}')
    ax.set_xlabel('Misclassified samples')
    ax.set_ylabel('True -> Predicted')
    save_fig(f'{prefix}_{split_name}_top_confusions_bar', fig)
    plt.show()
    return conf_df


def per_class_report_outputs(y_true, y_pred, prefix, title):
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(n_classes)),
        target_names=list(class_names),
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label_id, label_name in enumerate(class_names):
        item = report[str(label_name)]
        rows.append({
            'label_id': label_id,
            'label': label_name,
            'precision': item['precision'],
            'recall': item['recall'],
            'f1_score': item['f1-score'],
            'support': item['support'],
        })
    report_df = pd.DataFrame(rows)
    save_table(report_df, f'{prefix}_per_class_classification_report_test')

    plot_df = report_df.sort_values('f1_score', ascending=True).copy()
    plot_df['label_display'] = plot_df.apply(
        lambda r: f"{int(r['label_id'])}: {short_label(r['label'], 22)}", axis=1
    )
    fig, ax = plt.subplots(figsize=(11, max(7, 0.38 * n_classes)))
    sns.barplot(data=plot_df, y='label_display', x='f1_score', color='#2a9d8f', ax=ax)
    ax.set_title(f'Per-Class Test F1: {title}')
    ax.set_xlabel('F1-score')
    ax.set_ylabel('Class')
    ax.set_xlim(0, 1)
    save_fig(f'{prefix}_per_class_f1_test', fig)
    plt.show()
    return report_df


def cluster_purity_outputs(test_details, prefix, title):
    count_table = pd.crosstab(test_details['cluster_id'], test_details['true_label'])
    save_table(count_table.reset_index(), f'{prefix}_cluster_label_counts_test')

    purity_rows = []
    for cluster_id, row in count_table.iterrows():
        total = int(row.sum())
        majority_label = row.idxmax()
        majority_count = int(row.max())
        purity_rows.append({
            'cluster_id': int(cluster_id),
            'cluster_size': total,
            'majority_label': majority_label,
            'majority_count': majority_count,
            'purity': majority_count / total if total else np.nan,
        })
    purity_df = pd.DataFrame(purity_rows).sort_values(['cluster_size', 'purity'], ascending=False)
    save_table(purity_df, f'{prefix}_cluster_purity_test')

    fig, ax = plt.subplots(figsize=(12, max(5, 0.25 * len(purity_df))))
    plot_df = purity_df.sort_values('cluster_size', ascending=True).copy()
    ax.barh(plot_df['cluster_id'].astype(str), plot_df['purity'], color='#264653')
    ax.set_title(f'Test Cluster Purity: {title}')
    ax.set_xlabel('Purity')
    ax.set_ylabel('Cluster ID')
    ax.set_xlim(0, 1)
    save_fig(f'{prefix}_cluster_purity_test', fig)
    plt.show()

    pct_table = pd.crosstab(test_details['cluster_id'], test_details['true_label'], normalize='index') * 100.0
    pct_table = pct_table.reindex(index=purity_df['cluster_id'])
    fig_w = min(28, max(13, 0.5 * n_classes + 6))
    fig_h = min(24, max(7, 0.25 * len(pct_table) + 4))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(
        pct_table,
        cmap='mako',
        linewidths=0.2,
        linecolor='white',
        cbar_kws={'label': 'Percent within cluster (%)'},
        ax=ax,
    )
    ax.set_title(f'Test Cluster Composition: {title}')
    ax.set_xlabel('True class')
    ax.set_ylabel('Cluster ID')
    ax.tick_params(axis='x', rotation=60, labelsize=12)
    save_fig(f'{prefix}_cluster_composition_heatmap_test', fig)
    plt.show()
    return purity_df

In [ ]:
def plot_metric_bars_for_feature(feature_name):
    sub = metrics_df[(metrics_df['split'] == 'test') & (metrics_df['feature_set'] == feature_name)].copy()
    metric_cols = ['ari', 'nmi', 'v_measure', 'hungarian_accuracy', 'macro_f1', 'weighted_f1', 'silhouette']
    plot_df = sub.melt(
        id_vars=['cluster_model'],
        value_vars=metric_cols,
        var_name='metric',
        value_name='score',
    )
    fig, ax = plt.subplots(figsize=(12, 6.5))
    sns.barplot(data=plot_df, x='metric', y='score', hue='cluster_model', ax=ax)
    ax.set_title(f'Test Metrics by Clustering Model: {model_display_name(feature_name)}')
    ax.set_xlabel('Metric')
    ax.set_ylabel('Score')
    ax.legend(title='Cluster model', bbox_to_anchor=(1.02, 1), loc='upper left')
    save_fig(f'{feature_name}_test_metrics_by_clusterer', fig)
    plt.show()


def sample_test_indices(max_n=PLOT_SAMPLE):
    idx = np.where(split_masks['test'])[0]
    if max_n is not None and len(idx) > max_n:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(idx, size=max_n, replace=False)
    return np.sort(idx)


def scatter_embedding(embedding, color_values, title, save_name, color_kind='label'):
    plot_df = pd.DataFrame({
        'x': embedding[:, 0],
        'y': embedding[:, 1],
        'color': color_values,
    })
    if color_kind == 'label':
        palette = sns.color_palette('husl', n_colors=n_classes)
    else:
        n_colors = max(3, len(np.unique(color_values)))
        palette = sns.color_palette('Spectral', n_colors=n_colors)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.scatterplot(
        data=plot_df,
        x='x',
        y='y',
        hue='color',
        palette=palette,
        s=12,
        alpha=0.82,
        linewidth=0,
        legend=False,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    save_fig(save_name, fig)
    plt.show()


def embedding_visualizations(feature_name, output, prefix, title):
    X = prepared[feature_name]['X_pca']
    idx = sample_test_indices()
    X_plot = X[idx]
    true_ids = y_all[idx]
    clusters = output['clusters'][idx]

    scatter_embedding(
        X_plot[:, :2],
        true_ids,
        f'PCA Test Projection by True Class: {title}',
        f'{prefix}_pca_true_classes_test',
        color_kind='label',
    )
    scatter_embedding(
        X_plot[:, :2],
        clusters,
        f'PCA Test Projection by Cluster: {title}',
        f'{prefix}_pca_clusters_test',
        color_kind='cluster',
    )

    if RUN_UMAP:
        import umap

        reducer = umap.UMAP(
            n_neighbors=15,
            min_dist=0.1,
            metric='euclidean',
            random_state=RANDOM_STATE,
        )
        umap_xy = reducer.fit_transform(X_plot)
        scatter_embedding(
            umap_xy,
            true_ids,
            f'UMAP Test Projection by True Class: {title}',
            f'{prefix}_umap_true_classes_test',
            color_kind='label',
        )
        scatter_embedding(
            umap_xy,
            clusters,
            f'UMAP Test Projection by Cluster: {title}',
            f'{prefix}_umap_clusters_test',
            color_kind='cluster',
        )

    if RUN_TSNE and len(X_plot) >= 10:
        from sklearn.manifold import TSNE

        perplexity = min(30, max(5, (len(X_plot) - 1) // 3))
        tsne_xy = TSNE(
            n_components=2,
            perplexity=perplexity,
            init='pca',
            learning_rate='auto',
            random_state=RANDOM_STATE,
        ).fit_transform(X_plot)
        scatter_embedding(
            tsne_xy,
            true_ids,
            f't-SNE Test Projection by True Class: {title}',
            f'{prefix}_tsne_true_classes_test',
            color_kind='label',
        )
        scatter_embedding(
            tsne_xy,
            clusters,
            f't-SNE Test Projection by Cluster: {title}',
            f'{prefix}_tsne_clusters_test',
            color_kind='cluster',
        )

In [ ]:
for feature_name in prepared.keys():
    plot_metric_bars_for_feature(feature_name)

    key = best_keys[feature_name]
    output = cluster_outputs[key]
    cluster_model_name = key[1]
    title = f'{model_display_name(feature_name)} + {cluster_model_name}'
    prefix = f'{feature_name}_{cluster_model_name}'

    split_detail_tables = {}
    split_confusion_matrices = {}

    for split in ['valid', 'test']:
        mask = split_masks[split]
        y_true_split = y_all[mask]
        y_pred_split = output['predicted_labels'][mask]
        clusters_split = output['clusters'][mask]

        split_details = df.loc[mask, [
            'split',
            'image_name',
            'label',
            'label_id',
            'image_path',
            'width',
            'height',
            'area',
            'aspect_ratio',
        ]].copy()
        split_details = split_details.rename(columns={'label': 'true_label'})
        split_details['cluster_id'] = clusters_split
        split_details['predicted_label_id'] = y_pred_split
        split_details['predicted_label'] = [class_names[i] if 0 <= i < n_classes else 'unmapped' for i in y_pred_split]
        split_details['correct_after_mapping'] = split_details['label_id'].values == split_details['predicted_label_id'].values
        save_table(split_details, f'{prefix}_{split}_predictions_with_clusters')

        cm, cm_pct = confusion_outputs(y_true_split, y_pred_split, prefix, title, split_name=split)
        split_detail_tables[split] = split_details
        split_confusion_matrices[split] = cm

    top_confusions(split_confusion_matrices['test'], prefix, title, split_name='test')
    per_class_report_outputs(
        y_all[split_masks['test']],
        output['predicted_labels'][split_masks['test']],
        prefix,
        title,
    )
    cluster_purity_outputs(split_detail_tables['test'], prefix, title)
    embedding_visualizations(feature_name, output, prefix, title)

In [ ]:
comparison_cols = [
    'feature_model',
    'cluster_model',
    'selected_k',
    'n_clusters_found',
    'ari',
    'nmi',
    'ami',
    'homogeneity',
    'completeness',
    'v_measure',
    'fowlkes_mallows',
    'hungarian_accuracy',
    'macro_precision',
    'macro_recall',
    'macro_f1',
    'weighted_f1',
    'silhouette',
    'davies_bouldin',
    'calinski_harabasz',
]

paper_main = best_summary_df[comparison_cols].sort_values('macro_f1', ascending=False).reset_index(drop=True)
display(paper_main)
save_table(paper_main, 'paper_main_unsupervised_test_results')

plot_df = paper_main.melt(
    id_vars=['feature_model', 'cluster_model'],
    value_vars=['ari', 'nmi', 'v_measure', 'hungarian_accuracy', 'macro_f1', 'weighted_f1'],
    var_name='metric',
    value_name='score',
)
plot_df['method'] = plot_df['feature_model'] + '\n' + plot_df['cluster_model']

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=plot_df, x='metric', y='score', hue='method', ax=ax)
ax.set_title('Final Test Comparison: Best Clusterer per Feature Extractor')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend(title='Method', bbox_to_anchor=(1.02, 1), loc='upper left')
save_fig('final_best_feature_extractor_comparison_bar', fig)
plt.show()

heat = paper_main.set_index(['feature_model', 'cluster_model'])[
    ['ari', 'nmi', 'v_measure', 'hungarian_accuracy', 'macro_f1', 'weighted_f1']
]
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='viridis', linewidths=0.4, ax=ax)
ax.set_title('Final Test Comparison Heatmap')
save_fig('final_best_feature_extractor_comparison_heatmap', fig)
plt.show()

In [ ]:
def json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)

summary = {
    'dataset_dir': str(DATASET_DIR),
    'results_dir': str(RESULTS_DIR),
    'n_images': int(len(df)),
    'split_counts': {str(k): int(v) for k, v in df['split'].astype(str).value_counts().to_dict().items()},
    'n_classes': int(n_classes),
    'fixed_n_clusters': int(N_CLUSTERS),
    'feature_extractors': [cfg['display_name'] for cfg in FEATURE_CONFIGS],
    'selected_k_by_feature': {k: int(v) for k, v in selected_k_by_feature.items()},
    'best_unsupervised_test_results': paper_main.to_dict(orient='records'),
}

with open(RESULTS_DIR / 'research_run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=json_default)

artifact_manifest = pd.DataFrame({
    'artifact_type': ['figure'] * len(list((RESULTS_DIR / 'figures').glob('*'))) + ['table'] * len(list((RESULTS_DIR / 'tables').glob('*'))),
    'path': [str(p) for p in sorted((RESULTS_DIR / 'figures').glob('*'))] + [str(p) for p in sorted((RESULTS_DIR / 'tables').glob('*'))],
})
display(artifact_manifest.head(30))
save_table(artifact_manifest, 'artifact_manifest')

print('Saved all results to:', RESULTS_DIR)
print('Main paper table:', RESULTS_DIR / 'tables' / 'paper_main_unsupervised_test_results.csv')
print('High-resolution figures:', RESULTS_DIR / 'figures')